# IoTパート　RaspberryPiセットアップ

このNotebookは，情報科学演習で使うRaspberry Pi（Raspi）のセットアップ及びPCアクセス手順をまとめたものです．  
基本的には，以下の流れで進めます．

1. Raspiのセットアップ
2. Raspiにアクセスする
3. OpenCVでカメラを動かす

## 動作確認済みRaspi OS
Version: Debian GNU/Linux 12 (bookworm)で動作確認済みです．
```bash
$ lsb_release -a
No LSB modules are available.  
Distributor ID: Debian
Description: Debian GNU/Linux 12 (bookworm)
Release: 12
Codename: bookworm
```

## 1. RasPi：ソフトウェアセッティング

### 1.1 Raspi OSのインストール
始めにOSのセットアップを行います．  
Raspberry Pi OSのインストーラはRaspberry Pi Imagerを用いることで簡単に作成できます．  

以下のリンクよりダウンロード&インストールしてください．  
#### [Raspberry Pi Imager](https://www.raspberrypi.com/software/)

インストール手順
1. 書き込み用MicroSDカードをPCに挿入する
2. Raspberry Pi Imagerを起動する
3. OSを選択する
4. 書き込み用MicroSDカードを選択する
5. 指示に従ってOSイメージを書き込む

#### 書き込み時設定（02_pinode_setupに進む場合は02の1.1を先に参照すること）
- OS : Raspberry Pi OS(64bit)
- ホスト名 : XX
- ユーザー名とパスワードを設定する
  - ユーザ名 : XX
  - パスワード : XX
- Wi-Fiを設定する
  - SSID：XX
  - パスワード：XX
  - Wi-Fiを使う国：JP
- ロケール設定をする 
  - タイムゾーン : Asia/Tokyo
  - キーボードレイアウト: jp

### 1.2 Raspiの初期設定

必要物：書き込み済みMicroSDカード，Raspi本体，電源アダプタ，HDMIケーブル，モニタ，キーボード，マウス  
インストールが完了したら，RaspiにMicroSDカードを挿入して起動します．  
初回起動時は，Raspi OSのセットアップウィザードが表示されるので，指示に従ってセットアップを完了させてください．  
完了後，PCからRaspiにアクセスできるようにするための設定を行います．  

#### IPアドレスの固定
RaspiのIPアドレスを固定するため，以下の手順で設定を行います．
1. ターミナルを開く
2. 以下のコマンドを入力してネットワークインターフェースの設定ファイルを編集する（以下は例）
```bash
sudo nmcli con mod “Wired connection 1” ipv4.addresses “192.168.*.*/24“
sudo nmcli con mod "Wired connection 1" ipv4.method manual
sudo nmcli con up "Wired connection 1“
sudo nmcli con mod "Wired connection 1" ipv4.gateway 192.168.*.1 
sudo nmcli con mod "Wired connection 1" ipv4.dns 8.8.8.8
```

#### SSH・I2C・SPI有効化
RaspiでSSH，I2CやSPIを使用するため，以下の手順で設定を行います．
1. ターミナルを開く
2. 以下のコマンドを入力してRaspiの設定ツールを起動する
```bash
sudo raspi-config
```
Interfacing Options -> SSH（I4 I2C，I3 SPI）と進み，YESを選択します．

## 2. Raspiにアクセス
Raspiのセットアップが完了したら，PCからRaspiにアクセスできるようにするための設定を行います．

### PowerShellを用いたSSH接続
1. PowerShellを起動する
2. 以下のコマンドを入力してRaspiにSSH接続する
```powershell
ssh user@192.168.*.*
```
3. パスワードを入力して接続する

### VScode Remote-SSHを用いたSSH接続（複数回アクセスする場合）
1. VScodeからRemote-SSH拡張機能をインストールする
2. 上部中央タブから「ホストに接続する」→「SSHホストを構成する」→「~/.ssh/config」を選択する
3. 以下の内容を追加する（例）
    ```text
    Host raspi
        HostName 192.168.*.*
        User user
    ```
4. 上部中央タブから「ホストに接続する」→「raspi」を選択する
5. パスワードを入力して接続する（もしくは公開鍵設定）

## 3. OpenCVでカメラを動かす
Raspiにアクセスできるようになったら，OpenCVを用いてカメラを動かすための設定を行います．
先にRaspiのUSBにカメラを接続してから実施します．
1. ターミナルを開く
2. 以下のコマンドを入力してOpenCVをインストールする
```bash
sudo apt update
sudo apt install python3-opencv
```
3. ファイルを作成してカメラを動かすコードを記述する
```bash
nano camera.py
```
```python
import cv2
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()
```
4. ファイルを保存して終了する
5. 以下のコマンドを入力してカメラを動かす
```bash
python3 camera.py
```

## トラブルシューティング
### 時刻設定
時刻があっていない場合，出力の際に指定する時刻のずれが生じる可能性があります．以下のコマンドで時刻を確認し，必要に応じてタイムゾーンを設定してください．

Raspiの時刻確認
```bash
$ date
Thu  6 Nov 05:37:36 GMT 2025

$ timedatectl
Local time: Thu 2025-11-06 05:38:42 GMT
Universal time: Thu 2025-11-06 05:38:42 UTC
RTC time: n/a
Time zone: Europe/London (GMT, +0000)
System clock synchronized: yes
NTP service: active
RTC in local TZ: no
```
タイムゾーンが間違っている場合
```bash
$ sudo timedatectl set-timezone Asia/Tokyo
```